In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier


# -------------------------
# 1. LOAD DATA
# -------------------------

df = pd.read_csv(
    "cibil_score.csv"
)

df.columns = (
    df.columns
    .str.lower()
    .str.strip()
)

df = df.drop(
    columns=["unnamed: 0"],
    errors="ignore"
)

df = (
    df.drop_duplicates()
      .reset_index(drop=True)
)


# -------------------------
# 2. SPECIAL MISSING VALUES
# Use only if -9999 means missing
# -------------------------

df = df.replace(
    -9999,
    np.nan
)


# -------------------------
# 3. FEATURES AND TARGET
# -------------------------

X = df.drop(
    columns=[
        "approved_flag",
        "credit_score",
        "prospectid"
    ],
    errors="ignore"
)

y = df["approved_flag"]


# -------------------------
# 4. ENCODE TARGET
# P1,P2,P3,P4 -> 0,1,2,3
# -------------------------

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print(
    dict(
        zip(
            label_encoder.classes_,
            label_encoder.transform(
                label_encoder.classes_
            )
        )
    )
)


# -------------------------
# 5. FEATURE TYPES
# -------------------------

numeric_features = (
    X.select_dtypes(
        include=np.number
    )
    .columns
    .tolist()
)

categorical_features = (
    X.select_dtypes(
        exclude=np.number
    )
    .columns
    .tolist()
)


# -------------------------
# 6. TRAIN-TEST SPLIT
# -------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)


# -------------------------
# 7. PREPROCESSING
# -------------------------

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),

        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)


# -------------------------
# 8. BASIC XGBOOST
# -------------------------

basic_model = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    eval_metric="mlogloss",

    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,

    random_state=42,
    n_jobs=-1
)

basic_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "model",
            basic_model
        )
    ]
)

basic_pipeline.fit(
    X_train,
    y_train
)

basic_pred = basic_pipeline.predict(
    X_test
)


# -------------------------
# 9. REGULARIZED XGBOOST
# -------------------------

regularized_model = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    eval_metric="mlogloss",

    n_estimators=300,
    learning_rate=0.05,

    max_depth=5,
    min_child_weight=3,

    gamma=0.1,

    subsample=0.8,
    colsample_bytree=0.8,

    reg_alpha=0.1,
    reg_lambda=1.0,

    random_state=42,
    n_jobs=-1
)

regularized_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "model",
            regularized_model
        )
    ]
)

regularized_pipeline.fit(
    X_train,
    y_train
)

reg_pred = regularized_pipeline.predict(
    X_test
)


# -------------------------
# 10. EVALUATION FUNCTION
# -------------------------

def evaluate_multiclass(
    name,
    y_true,
    y_pred
):

    return {
        "Model": name,

        "Accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "Precision Macro":
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "Recall Macro":
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "F1 Macro":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "F1 Weighted":
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0
            )
    }


results = pd.DataFrame(
    [
        evaluate_multiclass(
            "Basic XGBoost",
            y_test,
            basic_pred
        ),

        evaluate_multiclass(
            "Regularized XGBoost",
            y_test,
            reg_pred
        )
    ]
)

print(results)


# -------------------------
# 11. CONFUSION MATRIX
# -------------------------

print(
    "\nRegularized Confusion Matrix:"
)

print(
    confusion_matrix(
        y_test,
        reg_pred
    )
)


# -------------------------
# 12. CLASSIFICATION REPORT
# -------------------------

print(
    "\nClassification Report:"
)

print(
    classification_report(
        y_test,
        reg_pred,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

{'P1': np.int64(0), 'P2': np.int64(1), 'P3': np.int64(2), 'P4': np.int64(3)}
                 Model  Accuracy  Precision Macro  Recall Macro  F1 Macro  \
0        Basic XGBoost  0.805123         0.734978      0.700491  0.712568   
1  Regularized XGBoost  0.805902         0.735308      0.698780  0.710634   

   F1 Weighted  
0     0.791818  
1     0.791045  

Regularized Confusion Matrix:
[[ 919  241    1    0]
 [ 176 6001  239   24]
 [  36  786  448  221]
 [   0   49  220  907]]

Classification Report:
              precision    recall  f1-score   support

          P1       0.81      0.79      0.80      1161
          P2       0.85      0.93      0.89      6440
          P3       0.49      0.30      0.37      1491
          P4       0.79      0.77      0.78      1176

    accuracy                           0.81     10268
   macro avg       0.74      0.70      0.71     10268
weighted avg       0.79      0.81      0.79     10268



The model achieved about 80.6% accuracy, but accuracy alone is somewhat misleading because P2 is the majority class.
Macro F1 was around 0.71. 
The model performed very well for P2 and reasonably well for P1 and P4, but P3 had only 30% recall and was frequently misclassified as P2. Regularization did not significantly improve performance, 
so the next step should be to address class imbalance or tune the model specifically to improve minority-class performance